# Train diffusion — GPU slice B

**Kernel:** `conda env:.conda-diffusion` (one GPU / MIG slice per notebook)

**Queue (sequential):** `anisotropic` → `cosine`

Run this notebook on one slice; run the companion notebook on the other.
Keep `SEQUENTIAL = True`.

Flag bundles: `configs/train_flags/{name}.sh`  
Defaults match Gray **iterE** (batch **32** for MIG / microbatch 10; iterE used 50) plus wall-time fixes (validation_interval 500, fp16, max_steps 111000).  
Set `DRY_RUN = False` before launching.


In [1]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))

from configs.paths import DATA_ROOT, SCRATCH_TRAINING, describe_environment, ensure_layout
from configs.train_configs import TRAIN_CONFIGS, list_configs_table, resolve_flag_script

ensure_layout()
print(describe_environment())
print()
print(list_configs_table())

discover = {}
for p in [DATA_ROOT / "training" / "eaf_discover.json", APP_ROOT / "train" / "eaf_discover.json"]:
    if p.is_file():
        discover = json.loads(p.read_text())
        print("loaded", p)
        break

# Prefer EAF-discovered checkout if it has anisotropic support; else local synced tree.
_candidates = []
if discover.get("diffusion_root"):
    _candidates.append(Path(discover["diffusion_root"]))
_candidates.append(APP_ROOT / "train" / "diffusion-anomaly")

DIFFUSION_ROOT = None
for c in _candidates:
    if (c / "scripts" / "image_train.py").exists() and (
        c / "guided_diffusion" / "anisotropic_diffusion.py"
    ).exists():
        DIFFUSION_ROOT = c
        break
if DIFFUSION_ROOT is None:
    DIFFUSION_ROOT = _candidates[-1]
    print("WARNING: anisotropic_diffusion.py missing — anisotropic config will fail")

assert (DIFFUSION_ROOT / "scripts" / "image_train.py").exists(), DIFFUSION_ROOT
print("DIFFUSION_ROOT =", DIFFUSION_ROOT)


{'hostname': 'jupyter-munjung', 'on_eaf': True, 'app_root': '/exp/sbnd/app/users/munjung/anomaly-detection', 'data_root': '/exp/sbnd/data/users/munjung/anomaly-detection', 'data_root_exists': True, 'scratch_root': '/scratch/7DayLifetime/munjung/anomaly-detection', 'scratch_exists': True, 'scratch_icarus_exists': True, 'diffusion_code_root': '/exp/sbnd/app/users/munjung/anomaly-detection/train/diffusion-anomaly', 'default_checkpoint': '/exp/sbnd/data/users/gputnam/training-SBND/iterE/results/brats2update111000.pt', 'default_checkpoint_exists': True}

name | schedule | x0 | aniso | flag
-----|----------|----|-------|-----
linear | linear | False | False | /exp/sbnd/app/users/munjung/anomaly-detection/configs/train_flags/linear.sh
cosine | cosine | False | False | /exp/sbnd/app/users/munjung/anomaly-detection/configs/train_flags/cosine.sh
ramp | ramp | False | False | /exp/sbnd/app/users/munjung/anomaly-detection/configs/train_flags/ramp.sh
anisotropic | linear | False | True | /exp/sbnd/

In [2]:
# --- Run controls (slice B) ---
DRY_RUN = False          # False → actually launch training
SEQUENTIAL = True       # keep True on a single GPU slice
CONFIGS_TO_RUN = ['anisotropic', 'cosine']

DATA_DIR = Path(
    discover.get("scratch_anomaly", "/scratch/7DayLifetime/munjung/anomaly-detection")
) / "npz"
VAL_DIR = DATA_DIR  # same as train (matches Gray iterE); set a held-out dir if desired
CHARGE_SCALE = "1"  # iterE flags omitted this (default 1); weight_pixels=False
MAX_STEPS = "111000"  # ~iterE production ckpt; constant LR (lr_anneal_steps=0)

RUN_ROOT = SCRATCH_TRAINING / "diffusion"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("SLICE  ", 'B')
print("DATA_DIR", DATA_DIR, "exists=", DATA_DIR.exists())
print("RUN_ROOT", RUN_ROOT)
print("CONFIGS", CONFIGS_TO_RUN)
print("DRY_RUN", DRY_RUN)
print("MAX_STEPS", MAX_STEPS, "CHARGE_SCALE", CHARGE_SCALE)


SLICE   B
DATA_DIR /scratch/7DayLifetime/munjung/anomaly-detection/npz exists= True
RUN_ROOT /scratch/7DayLifetime/munjung/anomaly-detection/training/diffusion
CONFIGS ['anisotropic', 'cosine']
DRY_RUN False
MAX_STEPS 111000 CHARGE_SCALE 1


In [3]:
def build_train_command(config_name: str) -> tuple[Path, Path, str]:
    flag = resolve_flag_script(config_name)
    assert flag.is_file(), flag
    log_dir = RUN_ROOT / config_name
    log_dir.mkdir(parents=True, exist_ok=True)
    log_file = log_dir / "train.log"
    # Bash: source flags (with DATA_DIR override), set OPENAI_LOGDIR, run image_train.
    cmd = f'''
set -euo pipefail
cd "{DIFFUSION_ROOT}"
export PYTHONPATH="/exp/sbnd/app/users/munjung/anomaly-detection/_stubs:{DIFFUSION_ROOT}:$PYTHONPATH"
export PYTORCH_ALLOC_CONF="${{PYTORCH_ALLOC_CONF:-expandable_segments:False}}"
unset PYTORCH_CUDA_ALLOC_CONF
# Do not overwrite CUDA_VISIBLE_DEVICES — Jupyter/MIG already set it.
echo "[$(date -Is)] CUDA_VISIBLE_DEVICES=${{CUDA_VISIBLE_DEVICES:-<unset>}}"
echo "[$(date -Is)] PYTORCH_ALLOC_CONF=$PYTORCH_ALLOC_CONF"
export DATA_DIR="{DATA_DIR}"
export VAL_DIR="{VAL_DIR}"
export CHARGE_SCALE="{CHARGE_SCALE}"
source "{flag}"
# Ensure max_steps is present even if an imported flag bundle omits it
export IMAGE_TRAIN_FLAGS="$IMAGE_TRAIN_FLAGS --max_steps {MAX_STEPS}"
export OPENAI_LOGDIR="{log_dir}"
mkdir -p "$OPENAI_LOGDIR"
echo "[$(date -Is)] CONFIG=$CONFIG_NAME"
echo "[$(date -Is)] FLAG={flag}"
echo "[$(date -Is)] OPENAI_LOGDIR=$OPENAI_LOGDIR"
echo "[$(date -Is)] IMAGE_TRAIN_FLAGS=$IMAGE_TRAIN_FLAGS"
python scripts/image_train.py $IMAGE_TRAIN_FLAGS 2>&1 | tee -a "{log_file}"
'''
    return flag, log_dir, cmd


jobs = {}
for name in CONFIGS_TO_RUN:
    flag, log_dir, cmd = build_train_command(name)
    jobs[name] = {"flag": flag, "log_dir": log_dir, "cmd": cmd}
    print(f"\n=== {name} ===")
    print(" flag   ", flag)
    print(" log_dir", log_dir)



=== anisotropic ===
 flag    /exp/sbnd/app/users/munjung/anomaly-detection/configs/train_flags/anisotropic.sh
 log_dir /scratch/7DayLifetime/munjung/anomaly-detection/training/diffusion/anisotropic

=== cosine ===
 flag    /exp/sbnd/app/users/munjung/anomaly-detection/configs/train_flags/cosine.sh
 log_dir /scratch/7DayLifetime/munjung/anomaly-detection/training/diffusion/cosine


In [4]:
# Preview / launch
procs = {}
for name, job in jobs.items():
    print("\n" + "=" * 60)
    print(f"CONFIG {name}")
    if DRY_RUN:
        print("(dry-run) would run:")
        print(job["cmd"])
        continue

    wrapper = job["log_dir"] / "launch.sh"
    wrapper.write_text(job["cmd"])
    wrapper.chmod(0o755)
    if SEQUENTIAL:
        print("launching sequential:", name)
        ret = subprocess.run(["bash", str(wrapper)], cwd=str(DIFFUSION_ROOT))
        print(name, "exit", ret.returncode)
        if ret.returncode != 0:
            raise SystemExit(f"{name} failed with {ret.returncode}")
    else:
        print("launching background:", name)
        procs[name] = subprocess.Popen(["bash", str(wrapper)], cwd=str(DIFFUSION_ROOT))

if not DRY_RUN and not SEQUENTIAL and procs:
    print("waiting on", list(procs))
    for name, p in procs.items():
        rc = p.wait()
        print(name, "exit", rc)



CONFIG anisotropic
launching sequential: anisotropic
[2026-09-15T01:11:35+00:00] CUDA_VISIBLE_DEVICES=<unset>
[2026-09-15T01:11:35+00:00] PYTORCH_ALLOC_CONF=expandable_segments:False
[2026-09-15T01:11:35+00:00] CONFIG=anisotropic
[2026-09-15T01:11:35+00:00] FLAG=/exp/sbnd/app/users/munjung/anomaly-detection/configs/train_flags/anisotropic.sh
[2026-09-15T01:11:35+00:00] OPENAI_LOGDIR=/scratch/7DayLifetime/munjung/anomaly-detection/training/diffusion/anisotropic
[2026-09-15T01:11:35+00:00] IMAGE_TRAIN_FLAGS=--data_dir /scratch/7DayLifetime/munjung/anomaly-detection/npz --validation_dir /scratch/7DayLifetime/munjung/anomaly-detection/npz --charge_scale 1 --image_size 512 --num_channels 32 --class_cond False --num_res_blocks 2 --num_heads 8 --learn_sigma True --use_scale_shift_norm False --attention_resolutions 16,32 --channel_mult 1,2,4,8,8,8 --diffusion_steps 1000 --noise_schedule linear --rescale_learned_sigmas False --rescale_timesteps False --predict_xstart False --anisotropic_noise 

KeyboardInterrupt: 

## Monitor (slice B)

```bash
tail -f /scratch/7DayLifetime/munjung/anomaly-detection/training/diffusion/anisotropic/train.log
tail -f /scratch/7DayLifetime/munjung/anomaly-detection/training/diffusion/cosine/train.log
```

Learning curves: `train/03_LearningCurves.ipynb`.
